In [1]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv

Tue Oct 13 08:42:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Exercise 1

In [2]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int*)malloc(size); random_ints(a, N);
    b = (int*)malloc(size); random_ints(b, N);
    c = (int*)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Initialize timer
    dStart = cpuSecond();

    // Launch add() kernel on GPU with N blocks
    add<<<N, 1>>>(d_a, d_b, d_c);

    // Wait for the GPU to finish, THEN stop the clock
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);

    return 0;
}

Writing ex1_timed.cu


In [3]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken 0.0006001


Exercise 2

In [4]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 512
// Checks the value returned by a CUDA runtime call. On failure it prints the
// file, line number, error code and reason, then stops the program.
#define CHECK(call) \
 do { \
 const cudaError_t error = (call); \
 if (error != cudaSuccess) { \
 printf("Error: %s:%d, ", __FILE__, __LINE__); \
 printf("code:%d, reason: %s\n", error, \
 cudaGetErrorString(error)); \
 exit(1); \
 } \
 } while (0)
__global__ void add(int *a, int *b, int *c) {
 c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}
void random_ints(int *x, int size) {
 for (int i = 0; i < size; i++)
 x[i] = rand() % 100;
}
double cpuSecond() {
 struct timeval tp;
 gettimeofday(&tp, NULL);
 return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}
int main(void) {
 int *a, *b, *c; // host copies of a, b, c
 int *d_a, *d_b, *d_c; // device copies of a, b, c
 int size = N * sizeof(int);
 double dStart, dElaps;
 // TODO 1: wrap each of the three cudaMalloc calls in CHECK(...)
 cudaMalloc((void **)&d_a, size);
 cudaMalloc((void **)&d_b, size);
 cudaMalloc((void **)&d_c, size);
 a = (int *)malloc(size); random_ints(a, N);
 b = (int *)malloc(size); random_ints(b, N);
 c = (int *)malloc(size);
 // TODO 2: wrap both host-to-device cudaMemcpy calls in CHECK(...)
 cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
 cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);
 dStart = cpuSecond();
 add<<<N, 1>>>(d_a, d_b, d_c);
 // TODO 3: add CHECK(cudaGetLastError()); here, and wrap cudaDeviceSynchronize()
 cudaDeviceSynchronize();
 dElaps = cpuSecond() - dStart;
 // TODO 4: wrap the device-to-host cudaMemcpy in CHECK(...)
 cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
 for (int r = 0; r < 5; r++)
 printf("%d + %d = %d\n", a[r], b[r], c[r]);
 int errors = 0;
 for (int r = 0; r < N; r++)
 if (c[r] != a[r] + b[r]) errors++;
 printf("Verification: %d mismatches out of %d\n", errors, N);
 printf("Time taken - %.7f\n", dElaps);
 free(a); free(b); free(c);
 // TODO 5: wrap the three cudaFree calls in CHECK(...)
 cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
 return 0;
}

Writing ex2_check.cu


In [5]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0000980


Exercise 3

In [6]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define WIDTH 800 // columns
#define HEIGHT 600 // rows
#define CHECK(call) \
 do { \
 const cudaError_t error = (call); \
 if (error != cudaSuccess) { \
 printf("Error: %s:%d, ", __FILE__, __LINE__); \
 printf("code:%d, reason: %s\n", error, \
 cudaGetErrorString(error)); \
 exit(1); \
 } \
 } while (0)
double cpuSecond() {
 struct timeval tp;
 gettimeofday(&tp, NULL);
 return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}
// Kernel from the lecture slide "Source Code of a PictureKernel", adapted
// for 8-bit pixels: scale every pixel value by 2.0, but never exceed 255.
__global__ void PictureKernel(unsigned char *d_Pin, unsigned char *d_Pout,
 int height, int width) {
 // Calculate the row # of the d_Pin and d_Pout element
 int Row = blockIdx.y * blockDim.y + threadIdx.y;
 // Calculate the column # of the d_Pin and d_Pout element
 int Col = blockIdx.x * blockDim.x + threadIdx.x;
 // each thread computes one element of d_Pout if in range
 if ((Row < height) && (Col < width)) {
 int v = 2 * d_Pin[Row * width + Col];
 d_Pout[Row * width + Col] = (v > 255) ? 255 : v;
 }
}
int main(void) {
 size_t count = (size_t)WIDTH * HEIGHT; // ONE flat array: 480,000 pixels
 size_t size = count * sizeof(unsigned char);
 unsigned char *h_in, *h_out; // host: original and updated picture
 unsigned char *d_in, *d_out; // device copies
 double dStart, dElaps;
 h_in = (unsigned char *)malloc(size);
 h_out = (unsigned char *)malloc(size);
 CHECK(cudaMalloc((void **)&d_in, size));
 CHECK(cudaMalloc((void **)&d_out, size));
 // TODO 1: give every pixel of h_in a random value between 0 and 255
 CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));
 // TODO 2: 16 threads per block in BOTH x and y
 dim3 threadsPerBlock(1, 1); // <-- replace this line
 // TODO 3: blocks needed in x (columns) and in y (rows), rounding UP
 dim3 numBlocks(1, 1); // <-- replace this line
 printf("Grid: %d x %d blocks of %d x %d threads\n",
 numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y);
 dStart = cpuSecond();
 // TODO 4: launch PictureKernel with <<<numBlocks, threadsPerBlock>>>
 CHECK(cudaGetLastError());
 CHECK(cudaDeviceSynchronize());
 dElaps = cpuSecond() - dStart;
 CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));
 // TODO 5: print the top-left 50 x 50 portion: first the original values,
 // then the updated values (use printf("%3d ", ...) and a newline per row)
 // TODO 6: check every pixel on the CPU (expected = 2 x original, capped at 255),
 // count the mismatches, and count how many pixels were capped
 printf("Time taken - %.7f\n", dElaps);
 free(h_in); free(h_out);
 CHECK(cudaFree(d_in)); CHECK(cudaFree(d_out));
 return 0;
}

Writing ex3_picture.cu


In [7]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture

Grid: 1 x 1 blocks of 1 x 1 threads
Time taken - 0.0000110


Exercise 4

In [11]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define NX 500    // x: columns
#define NY 500    // y: rows
#define NZ 500    // z: planes (depth)

#define CHECK(call)                                                          \
do {                                                                         \
    const cudaError_t error = (call);                                        \
    if (error != cudaSuccess) {                                              \
        printf("Error: %s:%d, ", __FILE__, __LINE__);                        \
        printf("code:%d, reason: %s\n", error, cudaGetErrorString(error));   \
        exit(1);                                                             \
    }                                                                        \
} while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Element-wise product: C[z][y][x] = A[z][y][x] * B[z][y][x]
__global__ void mulElem3D(int *a, int *b, int *c, int nx, int ny, int nz) {
    // TODO 1: the x, y and z coordinates of this thread
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    // TODO 2 & 3: guard and flat index computation
    if (x < nx && y < ny && z < nz) {
        size_t idx = ((size_t)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    size_t count = (size_t)NX * NY * NZ; // 125,000,000 elements
    size_t size = count * sizeof(int);   // 500 MB per array

    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    double dStart, dElaps;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int*)malloc(size); random_ints(a, count);
    b = (int*)malloc(size); random_ints(b, count);
    c = (int*)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // 8 x 8 x 8 = 512 threads per block
    dim3 threadsPerBlock(8, 8, 8);

    // TODO 4: blocks needed in x, y and z, rounding UP
    dim3 numBlocks((NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
                   (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z);

    printf("Grid: %d x %d x %d blocks\n", numBlocks.x, numBlocks.y, numBlocks.z);

    dStart = cpuSecond();

    // TODO 5: launch mulElem3D
    mulElem3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // TODO 6: print the last 1000 elements of C in memory order
    // C[499][498][0] to C[499][499][499]
    size_t startIndex = count - 1000;
    for (size_t idx = startIndex; idx < count; idx++) {
        int z = idx / (NX * NY);
        int rem = idx % (NX * NY);
        int y = rem / NX;
        int x = rem % NX;

        if (idx == startIndex || idx == count - 1) {
            printf("C[%d][%d][%d] = %d * %d = %d\n", z, y, x, a[idx], b[idx], c[idx]);
        }
    }

    // TODO 7: check all elements on the CPU and print mismatches
    size_t errors = 0;
    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) {
            errors++;
        }
    }

    printf("Verification: %zu mismatches out of %zu\n", errors, count);
    printf("Time taken %.7f\n", dElaps);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));

    return 0;
}

Overwriting ex4_volume.cu


In [12]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume

Grid: 63 x 63 x 63 blocks
C[499][498][0] = 76 * 15 = 1140
C[499][499][499] = 47 * 68 = 3196
Verification: 0 mismatches out of 125000000
Time taken 0.0081239
